# Figure 3 — LinkD-Pheno phenotypic validation

Drug-sensitivity × CRISPR concordance stratified by LinkD-Select.

## Data availability

All inputs for this notebook are **copied into** `For Reviewer/source_data/` (or shown from `illustrations/` when a panel cannot be recomputed).

- No Zenodo download is required.
- No paths outside `For Reviewer/` are used after packaging.
- See `DATA_AVAILABILITY.md` and `source_data/manifest.csv` for origins and checksums.

**Files used below** are listed in each panel section.

- `known_drug_rank_crispr_cancer_driver_role.csv`
- `matched_cells.csv`

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from linkd_repro import paths, style, io, illustrate
style.apply()
paths.ensure_output_dirs()
print("For Reviewer root:", paths.ROOT)
print("source_data OK:", paths.SOURCE.exists())

## Panel a — Framework schematic

In [ ]:
illustrate.show_panel('fig3_a', title='Panel a')

## Panel b — Cell lines across tissue lineages

In [ ]:

cr = io.read_crispr()
# disease column encodes tissue / cancer type
if "disease" in cr.columns:
    counts = cr.groupby("disease")["cell_line"].nunique().sort_values(ascending=False).head(13)
else:
    counts = pd.Series(dtype=float)
fig, ax = plt.subplots(figsize=(5.5, 3.2))
if len(counts):
    ax.barh(counts.index[::-1], counts.values[::-1], color="#4C72B0")
ax.set_xlabel("N cell lines")
ax.set_title("Fig 3b — cell lines by disease / lineage")
fig.tight_layout()
out = style.save_panel(fig, "fig3_b_tissue_counts", counts.rename("n_cell_lines").reset_index())
plt.show()
print(out)


## Panel c — Concordance vs Selectivity (25 oncology genes)

In [ ]:

cr = io.read_crispr()
# pick top 25 genes by known frequency
top_genes = cr.loc[cr["is_known"].astype(str).str.lower().isin(["1","true","yes","known"]), "Gene"].value_counts().head(25).index
sub = cr[cr["Gene"].isin(top_genes)].copy()
x = "landmark_correlation" if "landmark_correlation" in sub.columns else "AUC_corr"
fig, ax = plt.subplots(figsize=(4.2, 3.6))
ax.scatter(sub[x], sub["Selectivity_Score"], s=6, alpha=0.35, c="#4C72B0")
ax.set_xlabel(x)
ax.set_ylabel("Selectivity_Score")
ax.set_title("Fig 3c — concordance vs selectivity (25 genes)")
fig.tight_layout()
out = style.save_panel(fig, "fig3_c_scatter", sub[[x, "Selectivity_Score", "Gene", "Drug Chembl ID"]])
plt.show()
print(out)


## Panel d — All pairs with known overlay

In [ ]:

cr = io.read_crispr()
x = "landmark_correlation"
cr = cr.dropna(subset=[x, "Selectivity_Score"]).copy()
cr["known_flag"] = cr["is_known"].astype(str).str.lower().isin(["1","true","yes","known"])
samp = cr.sample(n=min(30000, len(cr)), random_state=0)
fig, ax = plt.subplots(figsize=(4.2, 3.6))
pred = samp[~samp["known_flag"]]
ax.scatter(pred[x], pred["Selectivity_Score"], s=2, alpha=0.15, c="#4C72B0", label="predicted")
k = cr[cr["known_flag"]]
if len(k):
    k = k.sample(n=min(5000, len(k)), random_state=0)
    ax.scatter(k[x], k["Selectivity_Score"], s=4, alpha=0.35, c=style.PALETTE["known"], label="known")
ax.legend(frameon=False)
ax.set_xlabel(x)
ax.set_ylabel("Selectivity_Score")
ax.set_title("Fig 3d — all pairs + known overlay")
fig.tight_layout()
out = style.save_panel(fig, "fig3_d_overlay", cr[[x, "Selectivity_Score", "known_flag"]].sample(n=min(50000, len(cr)), random_state=0))
plt.show()
print(out)


## Panel e — Breast-cancer tissue slice

In [ ]:

cr = io.read_crispr()
breast = cr[cr["disease"].astype(str).str.contains("breast", case=False, na=False)].copy()
x = "landmark_correlation"
fig, ax = plt.subplots(figsize=(4.2, 3.6))
if len(breast):
    ax.scatter(breast[x], breast["Selectivity_Score"], s=8, alpha=0.4)
ax.set_xlabel(x)
ax.set_ylabel("Selectivity_Score")
ax.set_title(f"Fig 3e — breast (n={breast['Gene'].nunique()} genes, {breast['cell_line'].nunique()} lines)")
fig.tight_layout()
out = style.save_panel(fig, "fig3_e_breast", breast[[x, "Selectivity_Score", "Gene", "cell_line"]])
plt.show()
print(out)


## Panel f — Cumulative recovery by Selectivity tier

In [ ]:

cr = io.read_crispr().dropna(subset=["Selectivity_Score", "landmark_correlation"])
cr["is_known"] = cr["is_known"].astype(str).str.lower().isin(["1","true","yes","known"])
cr["tier"] = pd.qcut(cr["Selectivity_Score"], 3, labels=["Tier3", "Tier2", "Tier1"])
# For each drug, rank genes by concordance; compute cumulative known recovery
def recovery_curve(g, ks=range(1, 51)):
    g = g.sort_values("landmark_correlation", ascending=False)
    known = g["is_known"].to_numpy()
    out = []
    for k in ks:
        out.append(known[:k].sum() / max(known.sum(), 1))
    return out
ks = list(range(1, 51))
fig, ax = plt.subplots(figsize=(4.2, 3.4))
rows = []
for tier in ["Tier1", "Tier2", "Tier3"]:
    sub = cr[cr["tier"] == tier]
    curves = []
    for _, g in sub.groupby("Drug Chembl ID"):
        if g["is_known"].sum() == 0:
            continue
        curves.append(recovery_curve(g, ks))
    if not curves:
        continue
    mean = np.mean(curves, axis=0)
    ax.plot(ks, mean, label=tier)
    for k, v in zip(ks, mean):
        rows.append({"tier": tier, "K": k, "mean_recovery": v})
ax.plot(ks, np.array(ks)/cr.groupby("Drug Chembl ID").size().median(), ls="--", color="gray", label="random-ish")
ax.legend(frameon=False)
ax.set_xlabel("K")
ax.set_ylabel("Mean fraction known recovered")
ax.set_title("Fig 3f — recovery by Selectivity tier")
fig.tight_layout()
out = style.save_panel(fig, "fig3_f_recovery", pd.DataFrame(rows))
plt.show()
print(out)


## Panels g–h — Discovery volcano and novel network (approximate)

In [ ]:

cr = io.read_crispr()
cr["is_known"] = cr["is_known"].astype(str).str.lower().isin(["1","true","yes","known"])
# volcano: x = concordance, y = -log10 FDR
if "FDR" in cr.columns:
    cr = cr.dropna(subset=["landmark_correlation", "FDR"])
    cr["neglog10p"] = -np.log10(cr["FDR"].clip(lower=1e-300))
else:
    cr["neglog10p"] = np.nan
novel = cr[(~cr["is_known"]) & (cr["landmark_correlation"] > 0.2) & (cr.get("FDR", 1) < 0.05)].copy()
fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.5))
ax = axes[0]
ax.scatter(cr["landmark_correlation"], cr["neglog10p"], s=2, alpha=0.1, c="#bbbbbb")
ax.scatter(novel["landmark_correlation"], novel["neglog10p"], s=8, alpha=0.5, c=style.PALETTE["linkd"])
ax.set_xlabel("landmark_correlation")
ax.set_ylabel("-log10 FDR")
ax.set_title("Fig 3g — discovery volcano")
# network-like degree plot for top novel pairs
ax = axes[1]
top = novel.nlargest(34, "Selectivity_Score")
if len(top):
    deg = pd.concat([top["Drug Chembl ID"], top["Gene"]]).value_counts().head(20)
    ax.barh(deg.index.astype(str)[::-1], deg.values[::-1], color=style.PALETTE["linkd"])
ax.set_title("Fig 3h — top nodes among 34 novel-like pairs")
fig.tight_layout()
out = style.save_panel(fig, "fig3_gh_volcano_network", top if len(novel) else cr.head(0))
plt.show()
print("novel-like pairs:", len(novel))
print(out)
